# Task 2 — Classical and Neural SGS Closure Baselines
**Owner: Sudip Kumar Paudel** · EPITA DSA 2026

Reproduces four baselines under the same data conditions as Task 1:

| Variant | Model | Ref |
|---------|-------|-----|
| V1 | Dynamic Smagorinsky (Germano identity) | Park & Choi JFM 2021 |
| V2 | WALE (Wall-Adapting Local Eddy-viscosity) | Nicoud & Ducros 1999 |
| V3 | Non-equivariant MLP, MSE loss | Beck et al. J.Comp.Phys 2019 |
| V4 | Energy-conserving MLP (Cholesky output) | van Gastelen et al. 2025 |

In [ ]:
# Colab setup — clone repo and install dependencies
import subprocess, os, sys
from pathlib import Path

subprocess.run(["git", "clone", "-b", "feature/data-pipeline",
                "https://github.com/Prerna2912/SHEAR.git", "/content/SHEAR"],
               capture_output=True)
subprocess.run(["pip", "install", "-q", "torch-geometric", "scipy", "e3nn"], check=True)

os.chdir("/content/SHEAR/notebooks")
REPO = Path("/content/SHEAR")
SRC  = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("Setup complete")


## 0. Setup

In [ ]:
# !pip install -q torch-geometric scipy
import sys, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch

REPO = Path('.').resolve()
if REPO.name == 'notebooks':
    REPO = REPO.parent
SRC = REPO / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RESULTS_CSV = REPO / 'results' / 'baselines_results.csv'
RESULTS_CSV.parent.mkdir(exist_ok=True)
print(f'Device: {DEVICE}')

## 1. Data Loading

In [ ]:
from data.jhtdb import generate_synthetic_les_data
from data.dataset import build_dataloaders

tau_field, grad_field = generate_synthetic_les_data(n_les=64, seed=0)
print(f'tau_field:  {tau_field.shape}')
print(f'grad_field: {grad_field.shape}')
print(f'tau std: {tau_field.std():.4f}   grad std: {grad_field.std():.4f}')

train_loader, val_loader, test_loader, stats = build_dataloaders(
    tau_field, grad_field,
    n_train=4000, n_val=500, n_test=500,
    patch_size=8, k_neighbours=26,
    batch_size=256, num_workers=0, seed=42,
)
print(f'Test batches: {len(test_loader)}')

# Collect test set as numpy arrays
grad_list, tau_list = [], []
for batch in test_loader:
    grad_list.append(batch.grad_full.numpy().reshape(-1, 3, 3))
    tau_list.append(batch.tau_full.numpy().reshape(-1, 3, 3))
grad_test = np.concatenate(grad_list, axis=0)   # [N_total, 3, 3]
tau_test  = np.concatenate(tau_list,  axis=0)   # [N_total, 3, 3]
print(f'Test nodes: {grad_test.shape[0]:,}')

## 2. V1 — Dynamic Smagorinsky

In [ ]:
from baselines.smagorinsky import DynamicSmagorinsky

smag = DynamicSmagorinsky(delta=1.0, cs_static=0.17)

# Validate first
r_smag = smag.validate(grad_u=grad_test, tau_true=tau_test)

# Predict on test set (sub-cube by sub-cube for dynamic Cs)
tau_smag = smag.predict(grad_test)   # [N_total, 3, 3]
print(f'Predicted tau shape: {tau_smag.shape}')

# Save predictions
np.save(str(REPO / 'results' / 'smagorinsky_test_preds.npy'), tau_smag)

In [ ]:
from evaluation.metrics import compute_all_metrics_np, save_results_csv

tau_smag_t = torch.from_numpy(tau_smag.astype(np.float32))
tau_test_t = torch.from_numpy(tau_test.astype(np.float32))
grad_test_t = torch.from_numpy(grad_test.astype(np.float32))

metrics_smag = compute_all_metrics_np(tau_smag_t, tau_test_t, grad_test_t)
save_results_csv(metrics_smag, 'smagorinsky', RESULTS_CSV)

print('\nDynamic Smagorinsky metrics:')
print(f'  Pearson r (mean):      {metrics_smag["pearson"]["mean"]:.4f}')
print(f'  Dissipation corr:      {metrics_smag["dissipation"]["correlation"]:.4f}')
print(f'  Backscatter fraction:  {metrics_smag["backscatter_fraction"]:.4f}  (classical models cannot backscatter)')

## 3. V2 — WALE

In [ ]:
from baselines.wale import WALE

wale = WALE(delta=1.0, Cw=0.325)
wale.validate()

tau_wale = wale.predict(grad_test)   # [N_total, 3, 3]
np.save(str(REPO / 'results' / 'wale_test_preds.npy'), tau_wale)

tau_wale_t = torch.from_numpy(tau_wale.astype(np.float32))
metrics_wale = compute_all_metrics_np(tau_wale_t, tau_test_t, grad_test_t)
save_results_csv(metrics_wale, 'wale', RESULTS_CSV)

print('\nWALE metrics:')
print(f'  Pearson r (mean):      {metrics_wale["pearson"]["mean"]:.4f}')
print(f'  Backscatter fraction:  {metrics_wale["backscatter_fraction"]:.4f}')

## 4. Filter-Width Sensitivity (Deeper Analysis)

In [ ]:
from scipy.stats import pearsonr

# Test all three filter widths
resolutions = [32, 64, 128]
pearson_smag = []
pearson_wale = []

for n_les in resolutions:
    tf, gf = generate_synthetic_les_data(n_les=n_les, seed=0)
    _, _, tl, _ = build_dataloaders(
        tf, gf, n_train=100, n_val=50, n_test=200,
        patch_size=8, k_neighbours=26, batch_size=256,
        num_workers=0, seed=42,
    )
    gl, tl_ = [], []
    for b in tl:
        gl.append(b.grad_full.numpy().reshape(-1, 3, 3))
        tl_.append(b.tau_full.numpy().reshape(-1, 3, 3))
    g_np = np.concatenate(gl); t_np = np.concatenate(tl_)

    s_pred = DynamicSmagorinsky(delta=1.0).predict(g_np)
    w_pred = WALE(delta=1.0).predict(g_np)

    r_s, _ = pearsonr(s_pred.reshape(-1), t_np.reshape(-1))
    r_w, _ = pearsonr(w_pred.reshape(-1), t_np.reshape(-1))
    pearson_smag.append(r_s)
    pearson_wale.append(r_w)
    print(f'n_les={n_les:3d}  Smag r={r_s:.4f}  WALE r={r_w:.4f}')

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(resolutions, pearson_smag, 'o-', label='Dynamic Smagorinsky', color='#1f77b4')
ax.plot(resolutions, pearson_wale, 's-', label='WALE', color='#d62728')
ax.set_xlabel('LES grid resolution (n³)'); ax.set_ylabel('Pearson r (all components)')
ax.set_title('Classical closure correlation vs filter width')
ax.legend(); ax.set_xscale('log'); ax.set_xticks(resolutions)
ax.set_xticklabels([f'{n}³' for n in resolutions])
plt.tight_layout()
plt.savefig(str(REPO / 'figures' / 'filter_width_sensitivity.png'), dpi=150)
plt.show()

## 5. V3 — Beck MLP (non-equivariant, MSE loss)

In [ ]:
from baselines.beck_mlp import BeckMLP, train_mlp, run_inference

beck = BeckMLP(input_dim=9, hidden_dim=256, output_dim=6, n_layers=3)
print(f'BeckMLP parameters: {beck.n_params:,}')

history_beck = train_mlp(
    beck, train_loader, val_loader,
    n_steps=10_000, lr=1e-3, weight_decay=1e-4, seed=42,
    log_every=2000, val_every=2000,
    out_dir=str(REPO / 'runs' / 'beck_mlp'),
    device=DEVICE,
)

In [ ]:
# Plot training curve
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(history_beck['train_loss'], alpha=0.5, label='Train')
steps, vals = zip(*history_beck['val_loss'])
ax.plot(steps, vals, 'o-', label='Val')
ax.set_title('Beck MLP training'); ax.set_yscale('log'); ax.legend()
plt.tight_layout(); plt.savefig(str(REPO / 'figures' / 'beck_mlp_curve.png'), dpi=150)
plt.show()

In [ ]:
# Load best checkpoint and evaluate
from baselines.beck_mlp import voigt_to_tau_3x3

ckpt = torch.load(str(REPO / 'runs' / 'beck_mlp' / 'best.pt'), map_location=DEVICE)
beck.load_state_dict(ckpt['model_state'])
beck = beck.to(DEVICE).eval()

tau_beck_voigt = run_inference(beck, grad_test.reshape(-1, 9), device=DEVICE)  # [N, 6]
tau_beck_t = voigt_to_tau_3x3(torch.from_numpy(tau_beck_voigt.astype(np.float32)))
np.save(str(REPO / 'results' / 'beck_mlp_test_preds.npy'), tau_beck_t.numpy())

metrics_beck = compute_all_metrics_np(tau_beck_t, tau_test_t, grad_test_t)
save_results_csv(metrics_beck, 'beck_mlp', RESULTS_CSV)

print('\nBeck MLP metrics:')
print(f'  Pearson r (mean):      {metrics_beck["pearson"]["mean"]:.4f}')
print(f'  Dissipation corr:      {metrics_beck["dissipation"]["correlation"]:.4f}')
print(f'  Backscatter fraction:  {metrics_beck["backscatter_fraction"]:.4f}')

## 6. V4 — Energy-Conserving MLP (van Gastelen et al. 2025)

In [ ]:
from baselines.beck_mlp import EnergyConservingMLP

vg_mlp = EnergyConservingMLP(hidden_dim=256, n_layers=3)
print(f'EnergyConservingMLP parameters: {vg_mlp.n_params:,}')

history_vg = train_mlp(
    vg_mlp, train_loader, val_loader,
    n_steps=10_000, lr=1e-3, weight_decay=1e-4, seed=42,
    log_every=2000, val_every=2000,
    out_dir=str(REPO / 'runs' / 'van_gastelen_mlp'),
    device=DEVICE,
)

In [ ]:
ckpt = torch.load(str(REPO / 'runs' / 'van_gastelen_mlp' / 'best.pt'), map_location=DEVICE)
vg_mlp.load_state_dict(ckpt['model_state'])
vg_mlp = vg_mlp.to(DEVICE).eval()

tau_vg_voigt = run_inference(vg_mlp, grad_test.reshape(-1, 9), device=DEVICE)
tau_vg_t = voigt_to_tau_3x3(torch.from_numpy(tau_vg_voigt.astype(np.float32)))
np.save(str(REPO / 'results' / 'van_gastelen_mlp_test_preds.npy'), tau_vg_t.numpy())

metrics_vg = compute_all_metrics_np(tau_vg_t, tau_test_t, grad_test_t)
save_results_csv(metrics_vg, 'van_gastelen_mlp', RESULTS_CSV)

print('\nVan Gastelen MLP metrics:')
print(f'  Pearson r (mean):      {metrics_vg["pearson"]["mean"]:.4f}')
print(f'  Backscatter fraction:  {metrics_vg["backscatter_fraction"]:.6f}  (should be 0 by design!)')

## 7. Comparison — All Four Baselines

In [ ]:
from evaluation.metrics import load_results_csv

results = load_results_csv(RESULTS_CSV)

models_t2 = ['smagorinsky', 'wale', 'beck_mlp', 'van_gastelen_mlp']
metrics_show = [
    ('pearson.mean',           'Pearson r'),
    ('dissipation.correlation','Diss. corr'),
    ('alignment.mean_deg',     'Align (deg)'),
    ('backscatter_fraction',   'Backscatter'),
]

print(f'{"Model":<25}' + ''.join(f'{lbl:<16}' for _, lbl in metrics_show))
print('-' * 90)
for m in models_t2:
    if m not in results:
        continue
    row = f'{m:<25}'
    for key, _ in metrics_show:
        val = results[m].get(key, float('nan'))
        row += f'{val:<16.4f}' if isinstance(val, float) else f'{str(val):<16}'
    print(row)

print('\nKey observations:')
print('  - Classical models (Smagorinsky, WALE) have zero backscatter by design.')
print('  - Van Gastelen MLP also has zero backscatter (energy-conservation constraint).')
print('  - Beck MLP can represent backscatter but may overfit physically invalid patterns.')

## 8. Save to Results CSV for Task 1 Unified Table

Results are already written via `save_results_csv` above.
Task 1 notebook loads this same CSV to build the unified table.

In [ ]:
print(f'Results CSV location: {RESULTS_CSV}')
print(f'Models recorded: {list(load_results_csv(RESULTS_CSV).keys())}')